# Imports, paths, and dataset loading

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_paths = {
    "olist_customers": RAW_DIR / "olist_customers_dataset.csv",
    "olist_orders": RAW_DIR / "olist_orders_dataset.csv",
    "olist_order_items": RAW_DIR / "olist_order_items_dataset.csv",
    "olist_payments": RAW_DIR / "olist_order_payments_dataset.csv",
    "olist_reviews": RAW_DIR / "olist_order_reviews_dataset.csv",
}

datasets = {}
for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found: {path}")
    datasets[name] = pd.read_csv(path)

customers = datasets["olist_customers"].copy()
orders = datasets["olist_orders"].copy()
order_items = datasets["olist_order_items"].copy()
payments = datasets["olist_payments"].copy()
reviews = datasets["olist_reviews"].copy()

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers: (99441, 5)
olist_orders: (99441, 8)
olist_order_items: (112650, 7)
olist_payments: (103886, 5)
olist_reviews: (99224, 7)


# Customer purchase features

In [2]:
# Build one order-level value from payments.
# If payment information is unavailable, fall back to item price + freight.
payment_values = payments.groupby("order_id", as_index=False).agg(
    order_value=("payment_value", lambda values: values.sum(min_count=1))
)

item_values = order_items.assign(
    item_total=order_items["price"].fillna(0) + order_items["freight_value"].fillna(0)
).groupby("order_id", as_index=False).agg(
    item_order_value=("item_total", "sum")
)

order_values = payment_values.merge(
    item_values, on="order_id", how="outer", validate="one_to_one"
)

order_values["order_value"] = order_values["order_value"].combine_first(
    order_values["item_order_value"]
)
order_values = order_values[["order_id", "order_value"]]

customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    customer_orders["order_purchase_timestamp"], errors="coerce"
)

customer_features = customer_orders.groupby("customer_unique_id", as_index=False).agg(
    total_orders=("order_id", "nunique"),
    delivered_orders=("order_status", lambda values: (values == "delivered").sum()),
    cancelled_orders=("order_status", lambda values: (values == "canceled").sum()),
    last_order_date=("order_purchase_timestamp", "max")
)

customer_order_values = customer_orders[["customer_unique_id", "order_id"]].merge(
    order_values, on="order_id", how="left", validate="many_to_one"
)

average_order_values = customer_order_values.groupby(
    "customer_unique_id", as_index=False
).agg(average_order_value=("order_value", "mean"))

customer_features = customer_features.merge(
    average_order_values, on="customer_unique_id", how="left", validate="one_to_one"
)
customer_features["average_order_value"] = customer_features["average_order_value"].round(2)

print("Customer feature table shape:", customer_features.shape)
print("Customers without an available order value:",
      int(customer_features["average_order_value"].isna().sum()))
display(customer_features.head(10))


Customer feature table shape: (96096, 6)
Customers without an available order value: 0


,customer_unique_id,total_orders,delivered_orders,cancelled_orders,last_order_date,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,2018-05-10 10:56:27,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,2018-05-07 11:11:27,27.19
2,0000f46a3911fa3c0805444483337064,1,1,0,2017-03-10 21:05:03,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,2017-10-12 20:29:41,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,2017-11-14 19:45:42,196.89
5,0004bd2a26a76fe21f786e4fbd80607f,1,1,0,2018-04-05 19:33:16,166.98
6,00050ab1314c0e55a6ca13cf7181fecf,1,1,0,2018-04-20 12:57:23,35.38
7,00053a61a98854899e70ed204dd4bafe,1,1,0,2018-02-28 11:15:41,419.18
8,0005e1862207bf6ccc02e4228effd9a0,1,1,0,2017-03-04 23:32:12,150.12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,1,0,2018-03-12 15:22:12,129.76


# Delivery duration and delay

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

orders["actual_delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

# Positive delay means delivery occurred after the estimated date.
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

delivered_orders = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

delivered_orders["actual_delivery_days"] = delivered_orders["actual_delivery_days"].round(2)
delivered_orders["delivery_delay_days"] = delivered_orders["delivery_delay_days"].round(2)
delivered_orders["is_late_delivery"] = delivered_orders["delivery_delay_days"] > 0

delivery_summary = pd.DataFrame([{
    "delivered_orders_analysed": len(delivered_orders),
    "average_delivery_duration_days": round(delivered_orders["actual_delivery_days"].mean(), 2),
    "average_delay_days": round(delivered_orders["delivery_delay_days"].mean(), 2),
    "late_delivery_count": int(delivered_orders["is_late_delivery"].sum()),
    "late_delivery_percentage": round(delivered_orders["is_late_delivery"].mean() * 100, 2)
}])
display(delivery_summary)

,delivered_orders_analysed,average_delivery_duration_days,average_delay_days,late_delivery_count,late_delivery_percentage
0,96470,12.56,-11.18,7820,8.11


# Cancellation and order-status patterns

In [4]:
order_status_summary = (
    orders["order_status"].value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)
order_status_summary["percentage"] = (order_status_summary["order_count"] / len(orders) * 100).round(2)
display(order_status_summary)

# Cancellation percentage is an order-level metric, not a customer churn rate.

,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


# Review-score patterns

In [5]:
review_summary = (
    reviews["review_score"].value_counts(dropna=False).sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)
review_summary["percentage"] = (review_summary["review_count"] / len(reviews) * 100).round(2)
display(review_summary)
print("Average review score:", round(reviews["review_score"].mean(), 2))

,review_score,review_count,percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


Average review score: 4.09


# Delivery delay versus review score

In [6]:

# Keep one row per order in the reviews table
reviews_for_merge = (
    reviews[["order_id", "review_score"]]
    .dropna(subset=["order_id"])
    .groupby("order_id", as_index=False)
    .agg(review_score=("review_score", "mean"))
)

# Ensure delivered_orders has one row per order
delivered_orders_unique = delivered_orders.drop_duplicates(
    subset=["order_id"]
)

# Merge delivery information with order-level review scores
delivery_reviews = delivered_orders_unique.merge(
    reviews_for_merge,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

delivery_reviews["delivery_category"] = np.where(
    delivery_reviews["delivery_delay_days"] > 0,
    "Late Delivery",
    "On-Time or Early"
)

review_comparison = (
    delivery_reviews.groupby("delivery_category", as_index=False)
    .agg(
        total_reviews=("review_score", "count"),
        average_review_score=("review_score", "mean")
    )
)

review_comparison["average_review_score"] = (
    review_comparison["average_review_score"].round(2)
)

display(review_comparison)

print(
    "This is an observed association; it does not prove "
    "that delivery delay caused a lower score."
)


,delivery_category,total_reviews,average_review_score
0,Late Delivery,7655,2.57
1,On-Time or Early,88169,4.29


This is an observed association; it does not prove that delivery delay caused a lower score.


# Define the Olist inactivity proxy

In [7]:
customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    customer_orders["order_purchase_timestamp"], errors="coerce"
)
customer_features["last_order_date"] = pd.to_datetime(
    customer_features["last_order_date"], errors="coerce"
)

reference_date = customer_orders["order_purchase_timestamp"].max()
customer_features["days_since_last_order"] = (
    reference_date - customer_features["last_order_date"]
).dt.days

customer_features["retention_risk_proxy"] = np.select(
    [
        customer_features["days_since_last_order"].between(0, 30, inclusive="both"),
        customer_features["days_since_last_order"].between(31, 60, inclusive="both"),
        customer_features["days_since_last_order"].between(61, 90, inclusive="both"),
        customer_features["days_since_last_order"] > 90,
    ],
    ["recent", "monitor", "at_risk", "high_inactivity"],
    default="unknown"
)

print("Reference date:", reference_date)
display(
    customer_features["retention_risk_proxy"]
    .value_counts(dropna=False)
    .rename_axis("retention_risk_proxy")
    .reset_index(name="customer_count")
)
display(customer_features.head(10))

Reference date: 2018-10-17 17:30:18


,retention_risk_proxy,customer_count
0,high_inactivity,86432
1,at_risk,7990
2,monitor,1665
3,recent,9


,customer_unique_id,total_orders,delivered_orders,cancelled_orders,last_order_date,average_order_value,days_since_last_order,retention_risk_proxy
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,2018-05-10 10:56:27,141.90,160,high_inactivity
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,2018-05-07 11:11:27,27.19,163,high_inactivity
2,0000f46a3911fa3c0805444483337064,1,1,0,2017-03-10 21:05:03,86.22,585,high_inactivity
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,2017-10-12 20:29:41,43.62,369,high_inactivity
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,2017-11-14 19:45:42,196.89,336,high_inactivity
5,0004bd2a26a76fe21f786e4fbd80607f,1,1,0,2018-04-05 19:33:16,166.98,194,high_inactivity
6,00050ab1314c0e55a6ca13cf7181fecf,1,1,0,2018-04-20 12:57:23,35.38,180,high_inactivity
7,00053a61a98854899e70ed204dd4bafe,1,1,0,2018-02-28 11:15:41,419.18,231,high_inactivity
8,0005e1862207bf6ccc02e4228effd9a0,1,1,0,2017-03-04 23:32:12,150.12,591,high_inactivity
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,1,0,2018-03-12 15:22:12,129.76,219,high_inactivity


# Validate and save the baseline feature table

In [8]:
feature_columns = [
    "total_orders",
    "delivered_orders",
    "cancelled_orders",
    "average_order_value",
    "days_since_last_order",
]

print("Rows:", len(customer_features))
print("Unique customer IDs:", customer_features["customer_unique_id"].nunique())
print("Duplicate customer IDs:", int(customer_features["customer_unique_id"].duplicated().sum()))

display(customer_features[feature_columns].isna().sum().rename("missing_count").to_frame())
display(customer_features[feature_columns].describe().round(2))

output_path = PROCESSED_DIR / "olist_customer_features.csv"
customer_features.to_csv(output_path, index=False)

print("Saved feature table to:", output_path)
print("Saved rows:", len(customer_features))
print("Saved columns:", len(customer_features.columns))

Rows: 96096
Unique customer IDs: 96096
Duplicate customer IDs: 0


,missing_count
total_orders,0
delivered_orders,0
cancelled_orders,0
average_order_value,0
days_since_last_order,0


,total_orders,delivered_orders,cancelled_orders,average_order_value,days_since_last_order
count,96096.00,96096.00,96096.00,96096.00,96096.00
mean,1.03,1.00,0.01,161.40,287.74
std,0.21,0.27,0.08,222.31,153.41
min,1.00,0.00,0.00,0.00,0.00
25%,1.00,1.00,0.00,62.46,163.00
50%,1.00,1.00,0.00,105.83,268.00
75%,1.00,1.00,0.00,177.21,397.00
max,17.00,15.00,3.00,13664.08,772.00


Saved feature table to: c:\Users\Acer\Desktop\customer-churn-retention-mvp\data\processed\olist_customer_features.csv
Saved rows: 96096
Saved columns: 8


In [9]:
print("Shape:", customer_features.shape)
display(customer_features.head())
display(customer_features.isnull().sum())
print("Duplicate rows:", customer_features.duplicated().sum())

Shape: (96096, 8)


,customer_unique_id,total_orders,delivered_orders,cancelled_orders,last_order_date,average_order_value,days_since_last_order,retention_risk_proxy
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,2018-05-10 10:56:27,141.90,160,high_inactivity
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,2018-05-07 11:11:27,27.19,163,high_inactivity
2,0000f46a3911fa3c0805444483337064,1,1,0,2017-03-10 21:05:03,86.22,585,high_inactivity
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,2017-10-12 20:29:41,43.62,369,high_inactivity
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,2017-11-14 19:45:42,196.89,336,high_inactivity


customer_unique_id       0
total_orders             0
delivered_orders         0
cancelled_orders         0
last_order_date          0
average_order_value      0
days_since_last_order    0
retention_risk_proxy     0
dtype: int64

Duplicate rows: 0


In [10]:
# Investigate any remaining missing average order values
missing_order_values = customer_features[
    customer_features["average_order_value"].isna()
]
print("Customers with missing average_order_value:", len(missing_order_values))
display(missing_order_values)

# Final quality checks
print("Shape:", customer_features.shape)
print("Total missing values:", int(customer_features.isna().sum().sum()))
print("Duplicate rows:", int(customer_features.duplicated().sum()))
print(
    "Duplicate customer IDs:",
    int(customer_features["customer_unique_id"].duplicated().sum())
)

# Save the final feature table
output_path = PROCESSED_DIR / "olist_customer_features.csv"
customer_features.to_csv(output_path, index=False)
print("Saved feature table to:", output_path)


Customers with missing average_order_value: 0


,customer_unique_id,total_orders,delivered_orders,cancelled_orders,last_order_date,average_order_value,days_since_last_order,retention_risk_proxy


Shape: (96096, 8)
Total missing values: 0
Duplicate rows: 0
Duplicate customer IDs: 0
Saved feature table to: c:\Users\Acer\Desktop\customer-churn-retention-mvp\data\processed\olist_customer_features.csv


In [11]:
print(customer_features["retention_risk_proxy"].unique())

['high_inactivity' 'monitor' 'at_risk' 'recent']
